In [ ]:
import os

import re
import time
import pandas as pd
from googleapiclient.discovery import build


# =========================
# 1. API 키 입력
# =========================
API_KEY = os.environ["YOUTUBE_API_KEY"]

youtube = build("youtube", "v3", developerKey=API_KEY)


# =========================
# 2. 수집할 유튜브 영상 URL 입력
# =========================
video_urls = [
    "https://www.youtube.com/watch?v=xh_bGO9Z1wo",
    "https://www.youtube.com/watch?v=6gPJGgrrUFA&t=6s",
    "https://www.youtube.com/watch?v=wZLJQ5vKwmk",
    "https://www.youtube.com/watch?v=GqpTK0cnROs",
    "https://www.youtube.com/watch?v=uXX3KyajIIQ",
    "https://www.youtube.com/watch?v=TFYjq61VO70",
    "https://www.youtube.com/watch?v=wTcs3SJI4Pk",
    "https://www.youtube.com/watch?v=F-yv2-xeV2A",
    "https://www.youtube.com/watch?v=T0JQ-pnVW5o",
    "https://www.youtube.com/watch?v=iekZJyVrMec"

]
match_contexts = [
    "프리미어리그 29라운드: 맨유 2:1 패배. 맨유가 패배한 경기, 맨유와 상대팀 모두 득점한 경기",
    "프리미어리그 30라운드: 맨유 3:1 승리. 맨유와 상대팀 모두 득점한 경기.",
    "프리미어리그 31라운드: 맨유 2:2 무승부. 두팀 다 득점한 경기.",
    "프리미어리그 32라운드: 맨유 1:2 패배. 맨유가 패배한 경기, 맨유와 상대팀 모두 득점한 경기.",
    "프리미어리그 33라운드: 맨유 1:0 승리. 상대팀은 득점하지 못한 경기.",
    "프리미어리그 34라운드: 맨유 2:1 승리. 맨유와 상대팀 모두 득점한 경기.",
    "프리미어리그 35라운드: 맨유 3:2 승리. 맨유와 상대팀 모두 득점한 경기.",
    "프리미어리그 36라운드: 맨유 0:0 무승부. 두팀다 득점이 없는 경기.",
    "프리미어리그 37라운드: 맨유 3:2 승리. 맨유와 상대팀 모두 득점한 경기.",
    "프리미어리그 38라운드: 맨유 3:0 승리. 상대팀은 득점하지 못한 경기.",
]


# =========================
# 3. 영상 ID 추출 함수
# =========================
def get_video_id(url):
    patterns = [
        r"v=([^&]+)",
        r"youtu\.be/([^?&]+)",
        r"shorts/([^?&]+)",
        r"embed/([^?&]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    return None


# =========================
# 4. 댓글 최소 전처리 함수
# =========================
def clean_comment(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # HTML 태그 제거
    text = re.sub(r"<.*?>", " ", text)

    # URL 제거
    text = re.sub(r"http\S+|www\S+", " ", text)

    # 줄바꿈 제거
    text = text.replace("\n", " ").replace("\r", " ")

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================
# 5. 댓글 수집
# =========================
all_comments = []
seen_comment_ids = set()

# 비정상 무한 수집 방지용 안전장치
MAX_COMMENTS_PER_VIDEO = 5000

print("수집할 영상 개수:", len(video_urls))
print("=" * 60)

for idx, (url, match_context) in enumerate(zip(video_urls, match_contexts), start=1):
    video_id = get_video_id(url)

    if video_id is None:
        print(f"[스킵] 영상 ID 추출 실패: {url}")
        continue

    print(f"\n[{idx}/{len(video_urls)}] 댓글 수집 시작: {video_id}")

    page_count = 0
    video_comment_count = 0
    next_page_token = None

    try:
        while True:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                textFormat="plainText",
                order="relevance",
                pageToken=next_page_token
            )

            response = request.execute()
            page_count += 1

            items = response.get("items", [])
            new_count_this_page = 0

            for item in items:
                comment = item["snippet"]["topLevelComment"]
                comment_id = comment["id"]

                # 이미 수집한 댓글이면 스킵
                if comment_id in seen_comment_ids:
                    continue

                seen_comment_ids.add(comment_id)

                comment_data = comment["snippet"]

                original_comment = comment_data.get("textDisplay", "")
                clean_text = clean_comment(original_comment)

                all_comments.append({
                    "comment_id": comment_id,
                    "video_id": video_id,
                    "video_url": url,
                    "match_context": match_context,
                    "author": comment_data.get("authorDisplayName"),
                    "comment_original": original_comment,
                    "comment_clean": clean_text,
                    "like_count": comment_data.get("likeCount"),
                    "published_at": comment_data.get("publishedAt")
                })

                video_comment_count += 1
                new_count_this_page += 1

            print(
                f"[진행중] 영상 {video_id} | "
                f"{page_count}페이지 완료 | "
                f"이번 페이지 신규 {new_count_this_page}개 | "
                f"현재 {video_comment_count}개 수집"
            )

            # 이번 페이지에서 새 댓글이 하나도 없으면 반복 오류 가능성 있음
            if new_count_this_page == 0:
                print(f"[중단] 영상 {video_id} | 새 댓글이 0개라 반복 수집 방지 중단")
                break

            # 안전장치: 말도 안 되게 많이 수집되면 중단
            if video_comment_count >= MAX_COMMENTS_PER_VIDEO:
                print(f"[중단] 영상 {video_id} | {MAX_COMMENTS_PER_VIDEO}개 도달, 비정상 반복 방지")
                break

            next_page_token = response.get("nextPageToken")

            # 다음 페이지 없으면 정상 종료
            if not next_page_token:
                break

            time.sleep(0.1)

        print(f"[완료] 영상 {video_id} | 총 {video_comment_count}개 수집")

    except Exception as e:
        print(f"[에러 발생] 영상 {video_id}")
        print(e)
        continue


# =========================
# 6. 데이터프레임 생성 및 저장
# =========================
df = pd.DataFrame(all_comments)

print("\n" + "=" * 60)
print("전체 원본 수집 댓글 수:", len(df))

if len(df) > 0:
    df = df[df["comment_clean"].str.len() > 0].copy()

    # 혹시 모를 중복 최종 제거
    df = df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)

    print("중복 제거 후 댓글 수:", len(df))

    df.to_csv(
        "youtube_comments_clean2.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df.to_excel(
        "youtube_comments_clean2.xlsx",
        index=False
    )

    print("저장 완료")
    print("CSV 파일: youtube_comments_clean2.csv")
    print("Excel 파일: youtube_comments_clean2.xlsx")

else:
    print("수집된 댓글이 없습니다.")


# =========================
# 7. 확인
# =========================
df.head()